# Lab 13: Information Retrieval for RAG

## 1. What is Information Retrieval?

**Information retrieval (IR)** means finding relevant information from a collection of documents based on a user query.

**Example:** A student asks, *"How do transformers process words?"* The retrieval system searches course notes and returns the most relevant passages.

**Simple process:**

`Documents → Numerical representation → Compare with query → Rank → Top results`

## 2. Create a small document collection

Imagine these are short notes taken from an NLP course. Each string represents one document.

In [1]:
documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."]

for i, document in enumerate(documents):
    print(f"Document {i}: {document}")

Document 0: Transformers use self-attention to understand relationships between words.
Document 1: TF-IDF represents documents using word importance scores.
Document 2: Word2Vec learns dense vector representations of words.
Document 3: Named entity recognition identifies names, places, and organizations in text.
Document 4: Information retrieval finds relevant documents for a user query.
Document 5: Cosine similarity measures how similar two text vectors are.


## 3. Convert documents into TF-IDF vectors (sparse retrieval)

A computer cannot compare raw text directly using vector similarity. **TF-IDF** converts each document into a numerical vector.

- **TF** measures how frequently a word occurs in a document.
- **IDF** gives more importance to words that appear in fewer documents.

TF-IDF is a form of **sparse retrieval**: most word positions in the vectors are zero.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", document_vectors.shape)
print("First document vector:", document_vectors[0].toarray().round(2))

TF-IDF matrix shape: (6, 39)
First document vector: [[0.39 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.39 0.   0.   0.   0.   0.   0.39 0.   0.   0.
  0.   0.39 0.39 0.39 0.   0.   0.   0.   0.   0.   0.32]]


**Check:** The matrix has one row per document and one column per vocabulary word. A zero means that word has no TF-IDF weight in that document.

## 4. Search using a query

We use the **same TF-IDF vectorizer** to represent the user's query. Then we calculate **cosine similarity** between the query and every document.

- Higher similarity → more relevant according to TF-IDF.
- A similarity of 0 means the vectors do not share weighted terms.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

query = "How do transformers understand words?"
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, document_vectors)[0]

for i, score in enumerate(scores):
    print(f"Document {i}: {score:.3f}")

Document 0: 0.633
Document 1: 0.000
Document 2: 0.173
Document 3: 0.000
Document 4: 0.000
Document 5: 0.000


## 5. Retrieve the top-k documents

Rather than reading every document, the system returns the **k** documents with the highest similarity scores.

In [4]:
k = 3
ranked_ids = scores.argsort()[::-1][:k]

for rank, i in enumerate(ranked_ids, start=1):
    print(f"Rank {rank} | Score: {scores[i]:.3f}")
    print(documents[i])
    print()

Rank 1 | Score: 0.633
Transformers use self-attention to understand relationships between words.

Rank 2 | Score: 0.173
Word2Vec learns dense vector representations of words.

Rank 3 | Score: 0.000
Information retrieval finds relevant documents for a user query.



**Reading the last result.** Rank 3 is *"Information retrieval finds relevant documents for a
user query."* at a score of **0.000** — the loop prints the top `k` rows whether or not they
matched. `argsort()` sorts all six scores, three of which are zero, so the third slot is filled
by whichever zero-score document the sort happened to put first: the order among ties is an
implementation detail, not relevance. Part 2 fixes this with a `scores[i] > 0` filter.

## 6. Student exercises

**Exercise 1 — Change the query**  
Use `search()` to find notes about **named entity recognition**. Which document is ranked first?

**Exercise 2 — Change k**  
Search for `"document representation"` using `k=1` and then `k=3`. What changes?


**Before the exercises — `search()` does not exist in Part 1.** Both exercises below say "use
`search()`", but this notebook only ranks inline in section 5, with `k = 3` hard-coded; the
function itself is defined in Part 2. Defined here with the same signature, so Part 1 runs on
its own:

In [5]:
# The exercises call search(), but Part 1 never defines it - only the inline
# ranking in section 5 exists, and its k is fixed at 3. This is the same
# function Part 2 defines in its section 1.
def search(query, k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    for i in scores.argsort()[::-1][:k]:
        print(f"Score: {scores[i]:.3f} | {documents[i]}")

In [6]:
search("named entity recognition", k=3)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


#### Answer — Exercise 1 — document 3, score 0.625

| Rank | Document | Score |
|---|---|---|
| 1 | 3 — "Named entity recognition identifies names, places, and organizations in text." | **0.625** |
| 2 | 5 — "Cosine similarity measures how similar two text vectors are." | 0.000 |
| 3 | 4 — "Information retrieval finds relevant documents for a user query." | 0.000 |

Document **3** is ranked first, and it is the only real match: it is the only document containing
`named`, `entity` and `recognition`, so it is the only one whose vector shares non-zero terms with
the query. 0.625 is the cosine between the query's three weighted terms and that one document.

The other five documents score exactly **0.000** — they share no weighted term with the query.
Rows 2 and 3 are therefore padding, not results: the same blur between *rank* and *relevance*
that section 5 shows, and the reason Part 2 adds `search_filtered()`.

In [7]:
# k only changes how many rows come back, so run both values back to back.
for k in (1, 3):
    print(f"k={k}")
    search("document representation", k=k)
    print()

k=1
Score: 0.000 | Cosine similarity measures how similar two text vectors are.

k=3
Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
Score: 0.000 | Named entity recognition identifies names, places, and organizations in text.



#### Answer — Exercise 2 — k changes how many rows, not which ones are relevant

Every score is **0.000**, for both values of `k`, and that is the point of the exercise. The
query analyzer produces `document` and `representation`, but neither is a vocabulary term: the
corpus contains `documents` (documents 1 and 4), `representations` (document 2) and `represents`
(document 1). TF-IDF matches whole tokens and does not stem, so the query vector is all zeros and
its cosine with every document is zero.

| `k` | Rows printed | Scores |
|---|---|---|
| 1 | document 5 | 0.000 |
| 3 | documents 5, 4, 3 | 0.000, 0.000, 0.000 |

So raising `k` does **not** add relevant documents; it only pads the list with more arbitrary
zero-score ones. Which zero-score documents appear, and in what order, is decided by
`argsort()`'s tie-breaking, not by the corpus.

The lesson is the exact-token limitation the handout names in Part 2 section 2: `document` and
`documents` are different columns to this vectorizer. Stemming or lemmatising before vectorising,
or using a dense embedding model, would connect them.

## Summary

You built a small **information retrieval** system using TF-IDF and cosine similarity. It converts documents and queries into vectors, ranks documents by relevance, and returns the top results.

**Connection to RAG:** These retrieved documents can later be passed to a language model as supporting context for answer generation.